# Qwen3.5-0.8B Full Dataset Training on Colab

Train Qwen3.5-0.8B on full DialogSum dataset (12,460 samples) using LoRA.

**Usage:** Run cells 1→2→3→4→5 in order.

**Crash-safe:** All checkpoints and results are saved to Google Drive in real-time.
If Colab disconnects, just re-run Cell 2 + Cell 4 + Cell 5 to resume from where it stopped.

**Experiments:**
- exp0_qwen: Baseline (no length control)
- exp1_qwen: Length-controllable summarization
- exp1_multi_qwen: Multi-task (summarization + topic generation)

In [ ]:
# @title Cell 1: Install dependencies (~30s)
!pip install -q --upgrade transformers
!pip install -q datasets peft rouge-score==0.1.2 tqdm accelerate

# No need to install flash-attn on A100:
# PyTorch's built-in SDPA already uses optimized CUDA kernels on A100.
USE_FLASH_ATTN = False
print("Using PyTorch SDPA (optimized for A100). All done.")


In [ ]:
# @title Cell 2: Initialize (run after Cell 1 or after runtime restart)
import os, json, random, numpy as np, time, shutil
from pathlib import Path
from typing import Dict, List

import torch
from transformers import (
    AutoModelForCausalLM, AutoTokenizer,
    Trainer, TrainingArguments,
    __version__ as transformers_version,
)
from peft import LoraConfig, get_peft_model, TaskType, PeftConfig, PeftModel
from datasets import load_dataset
from tqdm import tqdm

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# A100 optimizations
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Transformers: {transformers_version}  |  Device: {device}", end="")
if torch.cuda.is_available():
    print(f" ({torch.cuda.get_device_name(0)}, {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB)")
else:
    print()
print(f"Flash Attention: {USE_FLASH_ATTN}")

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3.5-0.8B", padding_side="left")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f"Tokenizer: {len(tokenizer)} tokens")

print("Loading DialogSum...")
ds = load_dataset("knkarthick/dialogsum")
print(f"Train: {len(ds['train'])}  Val: {len(ds['validation'])}  Test: {len(ds['test'])}")

SHORT_MAX = 15; MEDIUM_MAX = 35
BUCKET_RANGES = {"SHORT": (5, 15), "MEDIUM": (16, 35), "LONG": (36, 999)}
LENGTH_INSTRUCTIONS = {
    "SHORT": "Write a very brief one-sentence summary of the dialogue in 5 to 15 words.",
    "MEDIUM": "Write a short summary of the dialogue in 16 to 35 words.",
    "LONG": "Write a detailed summary of the dialogue in more than 35 words.",
}
def get_length_bucket(s):
    wc = len(s.split())
    if wc <= SHORT_MAX: return "SHORT"
    if wc <= MEDIUM_MAX: return "MEDIUM"
    return "LONG"

print("Init complete.")

In [ ]:
# @title Cell 3: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# @title Cell 4: All functions (crash-safe)

MODEL_NAME = "Qwen/Qwen3.5-0.8B"
OUTPUT_BASE = Path("/content/drive/MyDrive/qwen_colab")
MAX_INPUT = 256
MAX_TARGET = 64
PROGRESS_FILE = OUTPUT_BASE / "progress.json"

# ── Progress tracking ─────────────────────────────────
def load_progress():
    """Load progress from Drive. Returns dict {exp_name: status}."""
    if PROGRESS_FILE.exists():
        with open(PROGRESS_FILE) as f:
            return json.load(f)
    return {}

def save_progress(progress):
    """Save progress to Drive immediately."""
    OUTPUT_BASE.mkdir(parents=True, exist_ok=True)
    with open(PROGRESS_FILE, "w") as f:
        json.dump(progress, f, indent=2)

def mark_status(exp_name, status, **extra):
    """Update one experiment's status and flush to Drive."""
    progress = load_progress()
    entry = {"status": status, "updated": time.strftime("%Y-%m-%d %H:%M:%S")}
    entry.update(extra)
    progress[exp_name] = entry
    save_progress(progress)

# ── Data preprocessing ────────────────────────────────
def build_training_pairs(dialogue, summary, topic, use_length_tokens, multitask):
    pairs = []
    if use_length_tokens:
        bucket = get_length_bucket(summary)
        instr = LENGTH_INSTRUCTIONS[bucket]
        prompt = f"Summarize the following dialogue.\nInstruction: {instr}\nDialogue:\n{dialogue}\nSummary: "
    else:
        prompt = f"Summarize the following dialogue.\nDialogue:\n{dialogue}\nSummary: "
    pairs.append((prompt, summary))
    if multitask:
        pairs.append((f"What is the topic of the following dialogue? Answer in a short phrase.\nDialogue:\n{dialogue}\nTopic: ", topic))
    return pairs

def preprocess_batch(batch, tok, use_length_tokens=False, multitask=False):
    all_input_ids, all_attention_mask, all_labels = [], [], []
    max_total = MAX_INPUT + MAX_TARGET
    for dialogue, summary, topic in zip(batch["dialogue"], batch["summary"], batch["topic"]):
        for prompt_text, target_text in build_training_pairs(dialogue, summary, topic, use_length_tokens, multitask):
            prompt_ids = tok.encode(prompt_text, add_special_tokens=False)
            target_ids = tok.encode(target_text + tok.eos_token, add_special_tokens=False)
            if len(prompt_ids) + len(target_ids) > max_total:
                prompt_ids = prompt_ids[-(max_total - len(target_ids)):]
            full_ids = prompt_ids + target_ids
            all_input_ids.append(full_ids)
            all_attention_mask.append([1] * len(full_ids))
            all_labels.append([-100] * len(prompt_ids) + list(target_ids))
    return {"input_ids": all_input_ids, "attention_mask": all_attention_mask, "labels": all_labels}

# ── Model loading ─────────────────────────────────────
def load_model():
    attn = "flash_attention_2" if USE_FLASH_ATTN else "sdpa"
    print(f"  Loading {MODEL_NAME} (attn={attn})...")
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.bfloat16, attn_implementation=attn)
    lora_cfg = LoraConfig(
        r=16, lora_alpha=32,
        target_modules=["q_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_dropout=0.05, bias="none", task_type=TaskType.CAUSAL_LM,
    )
    model = get_peft_model(model, lora_cfg)
    model.print_trainable_parameters()
    return model

# ── Data collator ─────────────────────────────────────
class CausalLMCollator:
    def __init__(self, pad_token_id):
        self.pad_token_id = pad_token_id
    def __call__(self, features):
        max_len = max(len(f["input_ids"]) for f in features)
        batch = {"input_ids": [], "attention_mask": [], "labels": []}
        for f in features:
            pad = max_len - len(f["input_ids"])
            batch["input_ids"].append(f["input_ids"] + [self.pad_token_id] * pad)
            batch["attention_mask"].append(f["attention_mask"] + [0] * pad)
            batch["labels"].append(f["labels"] + [-100] * pad)
        return {k: torch.tensor(v, dtype=torch.long) for k, v in batch.items()}

# ── Training (with resume support) ────────────────────
def find_latest_checkpoint(output_dir):
    """Find the latest checkpoint in output_dir."""
    output_dir = Path(output_dir)
    checkpoints = sorted(output_dir.glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[1]))
    return str(checkpoints[-1]) if checkpoints else None

def train_experiment(exp_name, use_length_tokens=False, multitask=False, epochs=5, batch_size=16):
    output_dir = OUTPUT_BASE / exp_name
    output_dir.mkdir(parents=True, exist_ok=True)

    # ── Skip if already completed ───────────────────────
    if (output_dir / "training_complete.json").exists():
        print(f"SKIP {exp_name}: already completed (delete training_complete.json to retrain)")
        return None

    print(f"\n{'='*60}")
    print(f"  {exp_name}  |  length={use_length_tokens}  multitask={multitask}  bs={batch_size}")
    print(f"{'='*60}")

    # ── Preprocess ──────────────────────────────────────
    print("Preprocessing...")
    train_tok = ds["train"].map(
        preprocess_batch, fn_kwargs={"tok": tokenizer, "use_length_tokens": use_length_tokens, "multitask": multitask},
        batched=True, batch_size=1000, remove_columns=ds["train"].column_names, desc="Train",
    )
    val_tok = ds["validation"].map(
        preprocess_batch, fn_kwargs={"tok": tokenizer, "use_length_tokens": use_length_tokens, "multitask": multitask},
        batched=True, batch_size=500, remove_columns=ds["validation"].column_names, desc="Val",
    )
    print(f"  train={len(train_tok)}  val={len(val_tok)}")

    # ── Model ───────────────────────────────────────────
    model = load_model()

    # ── Check for existing checkpoint (resume) ──────────
    latest_ckpt = find_latest_checkpoint(output_dir)
    if latest_ckpt:
        print(f"  RESUMING from {latest_ckpt}")
        mark_status(exp_name, "resuming", checkpoint=latest_ckpt)
    else:
        mark_status(exp_name, "training")

    # ── Training args ───────────────────────────────────
    args = TrainingArguments(
        output_dir=str(output_dir),
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        learning_rate=5e-5,
        warmup_steps=100,
        weight_decay=0.01,
        max_grad_norm=1.0,
        logging_steps=50,
        eval_strategy="steps",
        eval_steps=500,
        save_strategy="steps",
        save_steps=500,
        save_total_limit=3,  # Keep 3 checkpoints for safety
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        bf16=True,
        gradient_checkpointing=False,  # A100 has 40GB, no need to recompute
        dataloader_num_workers=4,
        dataloader_pin_memory=True,
        report_to="none",
        seed=SEED,
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_tok,
        eval_dataset=val_tok,
        processing_class=tokenizer,
        data_collator=CausalLMCollator(tokenizer.pad_token_id),
    )

    # ── Train (resume if checkpoint exists) ─────────────
    print("Training...")
    t0 = time.time()
    result = trainer.train(resume_from_checkpoint=latest_ckpt)
    elapsed = time.time() - t0

    # ── Save final model to Drive ───────────────────────
    trainer.save_model()
    tokenizer.save_pretrained(output_dir)

    # ── Save config + metrics to Drive ──────────────────
    loss = float(result.metrics.get("train_loss", float("nan")))
    cfg = {
        "experiment": exp_name, "model": MODEL_NAME,
        "use_length_tokens": use_length_tokens, "multitask": multitask,
        "train_samples": len(train_tok), "epochs": epochs, "batch_size": batch_size,
        "train_loss": loss, "training_time_sec": round(elapsed, 1),
    }
    with open(output_dir / "config.json", "w") as f:
        json.dump(cfg, f, indent=2)

    # Mark as complete (this file is the "lock" that prevents retraining)
    with open(output_dir / "training_complete.json", "w") as f:
        json.dump({"status": "done", "loss": loss, "elapsed_sec": round(elapsed, 1)}, f, indent=2)

    mark_status(exp_name, "trained", loss=loss)
    print(f"Done. Loss={loss:.4f}  Time={elapsed/60:.1f}min  Saved to {output_dir}")

    # ── Clean up checkpoints to save Drive space ────────
    for ckpt in sorted(output_dir.glob("checkpoint-*")):
        print(f"  Cleaning up {ckpt.name}...")
        shutil.rmtree(ckpt, ignore_errors=True)

    return trainer

# ── Evaluation ────────────────────────────────────────
def evaluate_experiment(exp_name, max_samples=500):
    model_dir = OUTPUT_BASE / exp_name
    if not model_dir.exists():
        print(f"SKIP {exp_name}: directory not found")
        return None

    with open(model_dir / "config.json") as f:
        cfg = json.load(f)
    use_len = cfg.get("use_length_tokens", False)

    attn = "flash_attention_2" if USE_FLASH_ATTN else "sdpa"
    peft_config = PeftConfig.from_pretrained(str(model_dir))
    base = AutoModelForCausalLM.from_pretrained(peft_config.base_model_name_or_path, dtype=torch.bfloat16, attn_implementation=attn)
    base.resize_token_embeddings(len(tokenizer))
    model = PeftModel.from_pretrained(base, str(model_dir)).to(device).eval()

    test_data = ds["test"].select(range(min(max_samples, len(ds["test"]))))
    prompts, refs, buckets = [], [], []
    for row in test_data:
        refs.append(row["summary"])
        buckets.append(get_length_bucket(row["summary"]))
        if use_len:
            b = get_length_bucket(row["summary"])
            instr = LENGTH_INSTRUCTIONS[b]
            prompts.append(f"Summarize the following dialogue.\nInstruction: {instr}\nDialogue:\n{row['dialogue']}\nSummary: ")
        else:
            prompts.append(f"Summarize the following dialogue.\nDialogue:\n{row['dialogue']}\nSummary: ")

    print(f"Generating {len(prompts)} summaries for {exp_name}...")
    preds = []
    for i in tqdm(range(0, len(prompts), 16)):
        enc = tokenizer(prompts[i:i+8], max_length=MAX_INPUT, truncation=True, padding=True, return_tensors="pt").to(device)
        plen = enc["input_ids"].shape[1]
        with torch.no_grad():
            out = model.generate(**enc, max_new_tokens=MAX_TARGET, pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id, do_sample=False)
        preds.extend(d.strip() for d in tokenizer.batch_decode(out[:, plen:], skip_special_tokens=True))

    from rouge_score import rouge_scorer as rs
    scorer = rs.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
    r1, r2, rL = [], [], []
    for pred, ref in zip(preds, refs):
        s = scorer.score(ref, pred)
        r1.append(s["rouge1"].fmeasure)
        r2.append(s["rouge2"].fmeasure)
        rL.append(s["rougeL"].fmeasure)

    results = {"experiment": exp_name, "n": len(preds),
               "rouge1": round(sum(r1)/len(r1)*100, 2),
               "rouge2": round(sum(r2)/len(r2)*100, 2),
               "rougeL": round(sum(rL)/len(rL)*100, 2)}

    if use_len:
        correct = sum(1 for p, b in zip(preds, buckets) if BUCKET_RANGES[b][0] <= len(p.split()) <= BUCKET_RANGES[b][1])
        results["length_acc"] = round(correct / len(preds), 4)
        for bname in ["SHORT", "MEDIUM", "LONG"]:
            idxs = [i for i, b in enumerate(buckets) if b == bname]
            if idxs:
                bc = sum(1 for i in idxs if BUCKET_RANGES[buckets[i]][0] <= len(preds[i].split()) <= BUCKET_RANGES[buckets[i]][1])
                results[f"len_acc_{bname.lower()}"] = round(bc / len(idxs), 4)

    with open(model_dir / "eval_results.json", "w") as f:
        json.dump(results, f, indent=2)
    mark_status(exp_name, "evaluated", rouge1=results["rouge1"], rougeL=results["rougeL"])

    print(f"\n{'='*50}")
    print(f"{exp_name}  (n={results['n']})")
    print(f"{'='*50}")
    print(f"  ROUGE-1: {results['rouge1']:.2f}  ROUGE-2: {results['rouge2']:.2f}  ROUGE-L: {results['rougeL']:.2f}")
    if "length_acc" in results:
        print(f"  Length Acc: {results['length_acc']*100:.1f}%")
    print(f"  Saved to {model_dir / 'eval_results.json'}")
    return results

# ── Run one experiment (train + eval) ─────────────────
def run_experiment(exp_name, use_length_tokens, multitask, epochs=5, batch_size=16):
    """Train + evaluate. Skips if already done. Resumes if interrupted."""
    # Check if already evaluated
    eval_path = OUTPUT_BASE / exp_name / "eval_results.json"
    if eval_path.exists():
        with open(eval_path) as f:
            r = json.load(f)
        print(f"SKIP {exp_name}: already evaluated (ROUGE-L={r['rougeL']:.2f})")
        return r

    # Train (will skip if already trained)
    train_experiment(exp_name, use_length_tokens, multitask, epochs, batch_size)

    # Evaluate
    return evaluate_experiment(exp_name)

print("Functions defined. Ready to train.")

## Cell 5: Run all experiments

This is crash-safe:
- Already completed experiments are **skipped**
- Interrupted training **resumes** from last checkpoint
- All results saved to Google Drive in real-time

If Colab disconnects: re-run Cell 2 + Cell 4 + Cell 5 to continue.

In [ ]:
# @title Cell 5: Run all experiments (crash-safe)
EXPERIMENTS = [
    ("exp0_qwen_full",      False, False),
    ("exp1_qwen_full",      True,  False),
    ("exp1_multi_qwen_full", True,  True),
]

print("="*60)
print("Starting experiment pipeline")
print("="*60)

# Show current status
progress = load_progress()
for exp_name, _, _ in EXPERIMENTS:
    status = progress.get(exp_name, {}).get("status", "not started")
    print(f"  {exp_name}: {status}")

# Run each experiment
all_results = {}
for exp_name, use_len, multitask in EXPERIMENTS:
    print(f"\n{'='*60}")
    result = run_experiment(exp_name, use_len, multitask, epochs=5, batch_size=32)
    if result:
        all_results[exp_name] = result

# ── Final summary ──────────────────────────────────────
print(f"\n{'='*70}")
print("ALL RESULTS")
print(f"{'='*70}")
for exp_name, _, _ in EXPERIMENTS:
    p = OUTPUT_BASE / exp_name / "eval_results.json"
    if p.exists():
        with open(p) as f:
            r = json.load(f)
        la = f"  LenAcc={r['length_acc']*100:.1f}%" if "length_acc" in r else ""
        print(f"  {exp_name}: R1={r['rouge1']:.2f}  R2={r['rouge2']:.2f}  RL={r['rougeL']:.2f}{la}")
    else:
        print(f"  {exp_name}: NOT DONE")

In [ ]:
# @title Cell 6: Check progress (can run anytime)
progress = load_progress()
print("Progress:")
for exp, info in progress.items():
    print(f"  {exp}: {info.get('status', 'unknown')}")

print("\nFiles on Drive:")
for exp_name in ["exp0_qwen_full", "exp1_qwen_full", "exp1_multi_qwen_full"]:
    d = OUTPUT_BASE / exp_name
    if d.exists():
        files = [f.name for f in d.iterdir() if not f.name.startswith("checkpoint")]
        print(f"  {exp_name}/: {', '.join(sorted(files))}")
    else:
        print(f"  {exp_name}/: not found")

In [ ]:
# @title Cell 7: Download results
# Download evaluation results (small, fast)
!cd /content/drive/MyDrive/qwen_colab && zip -r results.zip exp0_qwen_full/eval_results.json exp0_qwen_full/config.json exp1_qwen_full/eval_results.json exp1_qwen_full/config.json exp1_multi_qwen_full/eval_results.json exp1_multi_qwen_full/config.json progress.json 2>/dev/null
from google.colab import files
files.download('/content/drive/MyDrive/qwen_colab/results.zip')

In [ ]:
# @title Cell 8: Download model checkpoints (large, optional)
# Only run this if you want the model weights locally
!cd /content/drive/MyDrive/qwen_colab && zip -r models.zip exp0_qwen_full exp1_qwen_full exp1_multi_qwen_full -x "*/checkpoint-*"
files.download('/content/drive/MyDrive/qwen_colab/models.zip')